# Sampling control parameters

This notebook explains the different sampling strategies for control parameters that are
built in.

Note that as far as possible, priors are respected in sampling.

## Random sampling

From a `Space`, it is possible to do random sampling, which gives a sample according to
the `prior` distribution of the `Space`:

In [13]:
import ProcessOptimizer as po

# Define the space
space = po.Space(
    dimensions=[
        po.Real(1, 1000, prior='uniform'),
        po.Real(1, 1000, prior='log-uniform'),
        po.Integer(1, 10),
        po.Categorical(["cat", "dog", "elephant"]),
        po.Categorical(["A", "B", "C"], name="Tasks"),
    ]
)

# Generate random samples
random_sample_list = space.rvs(n_samples=10)

# Print the random samples
print("Sample from all tasks: ")
for i in range(10):
    print(f"{i}'th random sample: {random_sample_list[i]}")

space.dimensions[-1] = po.Task(["A", "B", "C"], active_task="C")

# Generate random samples
print("\nSample from the active task slice only: ")
random_sample_list = space.rvs(n_samples=10)

# Print the random samples
for i in range(10):
    print(f"{i}'th random sample: {random_sample_list[i]}")

Sample from all tasks: 
0'th random sample: [np.float64(260.5306932779365), 1.3036450306895186, 9, 'elephant', 'C']
1'th random sample: [np.float64(719.5321761464884), 234.60768067501044, 3, 'dog', 'B']
2'th random sample: [np.float64(53.56783972013898), 33.368157957967625, 3, 'elephant', 'A']
3'th random sample: [np.float64(810.6494635137993), 176.81628023561578, 6, 'dog', 'C']
4'th random sample: [np.float64(285.2275684008174), 6.968507796251866, 2, 'dog', 'A']
5'th random sample: [np.float64(314.4637035662815), 1.3699065022854673, 10, 'dog', 'B']
6'th random sample: [np.float64(807.6373459879984), 165.49719413304246, 10, 'elephant', 'B']
7'th random sample: [np.float64(298.50918697538594), 8.518286405580946, 3, 'elephant', 'B']
8'th random sample: [np.float64(289.182658551376), 2.0085177586818332, 3, 'cat', 'B']
9'th random sample: [np.float64(758.0110756934374), 2.317829207242637, 6, 'dog', 'B']

Sample from the active task slice only: 
0'th random sample: [np.float64(167.082008031

# Latin hypercube sampling

Random sampling is not a good starting point for doing Bayesian optimisation. It is better
to have the starting samples distributed over the dimensions in a controlled manner. This
is ensured by Latin Hypercube sampling, which provides samples that are guaranteed to be
equally distributed on each dimension with a uniform prior (but not on combinations of
dimensions). On dimensions with non-uniform priors, the prior is respected. This means
that the points sampled along a given dimension will have the distrubtion specified by the
prior.

In [2]:
# Generate LHS samples
print("Sample from all tasks: ")
space.dimensions[-1].use_active_task = False
LHS_sample_list = space.lhs(n=10)

# Print the LHS samples
for i in range(10):
    print(f"{i}'th LHS sample: {LHS_sample_list[i]}")
    
print("\nSample from the active task slice only: ")

space.dimensions[-1].use_active_task = True
LHS_sample_list = space.lhs(n=10)
# Print the LHS samples
for i in range(10):
    print(f"{i}'th LHS sample: {LHS_sample_list[i]}")

Sample from all tasks: 
0'th LHS sample: [np.float64(550.45), 22.387211385683397, 1, 'dog', 'B']
1'th LHS sample: [np.float64(650.35), 354.8133892335753, 10, 'cat', 'A']
2'th LHS sample: [np.float64(50.95), 5.62341325190349, 9, 'cat', 'B']
3'th LHS sample: [np.float64(750.25), 89.12509381337456, 5, 'elephant', 'B']
4'th LHS sample: [np.float64(350.65), 44.668359215096324, 8, 'elephant', 'A']
5'th LHS sample: [np.float64(250.75), 707.9457843841375, 3, 'dog', 'B']
6'th LHS sample: [np.float64(450.55), 177.82794100389225, 4, 'elephant', 'C']
7'th LHS sample: [np.float64(950.05), 11.22018454301963, 6, 'dog', 'C']
8'th LHS sample: [np.float64(150.85), 1.4125375446227544, 7, 'cat', 'A']
9'th LHS sample: [np.float64(850.15), 2.8183829312644537, 2, 'dog', 'C']

Sample from the active task slice only: 
0'th LHS sample: [np.float64(550.45), 22.387211385683397, 1, 'dog', 'C']
1'th LHS sample: [np.float64(650.35), 354.8133892335753, 10, 'cat', 'C']
2'th LHS sample: [np.float64(50.95), 5.6234132519

# Random states

Both random value sampling and Latin hypercube sampling supports taking a
random seed to allow for reproducible sampling. They support a variety of
formats, or `None` for true randomness.

Random value sampling is random by default, while Latin hypercube sampling is
pseudo-random. Note that randomising the Latin hypercube sampling results in
(mostly) different points being sampled, but the sampled values for each
dimension are the same.

In [3]:
# Define the space
space_definition = [[1., 10.], [1, 10], ["cat", "dog", "elephant"], ["A", "B", "C", {"C"}]]
space = po.Space(space_definition)

print("Sample from all tasks: ")
# Generate random samples and print them
for i in range(5):
    print(f"{i+1}'th random sample: {space.rvs(n_samples=1)}")

print("\n")

# Generate pseudo-random samples and print them
for i in range(5):
    print(f"{i+1}'th pseudo-random sample: {space.rvs(n_samples=1, random_state=2)}")

print("\n")

# Generate LHS samples and print them
print(f"First Latin hypercube sampling:  {space.lhs(n=5)}")
print(f"Second Latin hypercube sampling: {space.lhs(n=5)}")
print(f"LHS sampling with different seed:  {space.lhs(n=5, seed=2)}")

Sample from all tasks: 
1'th random sample: [[np.float64(5.516618120087536), 2, 'dog', 'C']]
2'th random sample: [[np.float64(8.999292517599693), 1, 'cat', 'C']]
3'th random sample: [[np.float64(7.030286139114125), 9, 'dog', 'C']]
4'th random sample: [[np.float64(5.3546428317240276), 3, 'elephant', 'C']]
5'th random sample: [[np.float64(7.601470378544546), 8, 'cat', 'C']]


1'th pseudo-random sample: [[np.float64(3.3545092082438477), 3, 'elephant', 'C']]
2'th pseudo-random sample: [[np.float64(3.3545092082438477), 3, 'elephant', 'C']]
3'th pseudo-random sample: [[np.float64(3.3545092082438477), 3, 'elephant', 'C']]
4'th pseudo-random sample: [[np.float64(3.3545092082438477), 3, 'elephant', 'C']]
5'th pseudo-random sample: [[np.float64(3.3545092082438477), 3, 'elephant', 'C']]


First Latin hypercube sampling:  [[np.float64(9.1), 8, 'dog', 'C'], [np.float64(5.5), 2, 'elephant', 'C'], [np.float64(7.3), 4, 'cat', 'C'], [np.float64(3.6999999999999997), 6, 'cat', 'C'], [np.float64(1.9), 10,

In [4]:
print("Sample from the active tasks slice of the space: ")
# Generate random samples and print them
for i in range(5):
    print(f"{i+1}'th random sample: {space.rvs(n_samples=1)}")

print("\n")

# Generate pseudo-random samples and print them
for i in range(5):
    print(f"{i+1}'th pseudo-random sample: {space.rvs(n_samples=1, random_state=2)}")

print("\n")

# Generate LHS samples and print them
print(f"First Latin hypercube sampling:  {space.lhs(n=5)}")
print(f"Second Latin hypercube sampling: {space.lhs(n=5)}")
print(f"LHS sampling with different seed:  {space.lhs(n=5, seed=2)}")

Sample from the active tasks slice of the space: 
1'th random sample: [[np.float64(4.4757626366866905), 6, 'cat', 'C']]
2'th random sample: [[np.float64(1.9940121586470423), 8, 'cat', 'C']]
3'th random sample: [[np.float64(3.7517382506249177), 6, 'cat', 'C']]
4'th random sample: [[np.float64(5.04593456956968), 4, 'cat', 'C']]
5'th random sample: [[np.float64(4.23310383021592), 8, 'dog', 'C']]


1'th pseudo-random sample: [[np.float64(3.3545092082438477), 3, 'elephant', 'C']]
2'th pseudo-random sample: [[np.float64(3.3545092082438477), 3, 'elephant', 'C']]
3'th pseudo-random sample: [[np.float64(3.3545092082438477), 3, 'elephant', 'C']]
4'th pseudo-random sample: [[np.float64(3.3545092082438477), 3, 'elephant', 'C']]
5'th pseudo-random sample: [[np.float64(3.3545092082438477), 3, 'elephant', 'C']]


First Latin hypercube sampling:  [[np.float64(9.1), 8, 'dog', 'C'], [np.float64(5.5), 2, 'elephant', 'C'], [np.float64(7.3), 4, 'cat', 'C'], [np.float64(3.6999999999999997), 6, 'cat', 'C'], 